In [5]:
# Para tratamiento de datos
import pandas as pd
import numpy as np
import re #para llamar a Expresiones Regulares y estandarizar el nombre de las columnas.
import time
import requests

# Para visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns

# Para poder visualizar todas las columnas de los DataFrames
pd.set_option('display.max_columns', None) 

# Trabajar con el sistema operativo y variables de entorno
import os 
from dotenv import load_dotenv
load_dotenv() #carga las variables del entorno .env; devuelve un true o false


# Conexión con MySQL
import mysql.connector
from mysql.connector import Error

# Gestión de los warnings
import warnings
warnings.filterwarnings("ignore")



In [3]:
# Cargamos las variables de entorno del archivo .env
load_dotenv()

# 1. Recuperamos la Key y el ID base
steam_key = os.getenv("steam_key")

# 2. Creamos la lista de los 21 IDs dinámicamente
lista_ids = []
for i in range(21):
    variable_name = f"steam_id_{i}"
    valor_id = os.getenv(variable_name)
    if valor_id:
        lista_ids.append(valor_id)

print(f"Se han cargado {len(lista_ids)} IDs desde el archivo .env")

Se han cargado 21 IDs desde el archivo .env


In [ ]:
load_dotenv()
steam_key = os.getenv("steam_key")

# Cargamos los 21 IDs desde el .env
ids_proyecto = [os.getenv(f"steam_id_{i}") for i in range(21)]
ids_proyecto = [i for i in ids_proyecto if i] # Limpiamos valores None

registros_juegos = []

print(f"--- Iniciando construcción del Dataset ({len(ids_proyecto)} usuarios) ---")

for s_id in ids_proyecto:
    # Usamos HTTPS y especificamos el formato JSON explícitamente
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        'key': steam_key,
        'steamid': s_id,
        'format': 'json',
        'include_appinfo': 1,
        'include_played_free_games': 1
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            # Accedemos a la estructura de la respuesta
            juegos = data.get('response', {}).get('games', [])
            
            for juego in juegos:
                juego['user_id'] = s_id
                registros_juegos.append(juego)
            
            print(f"✅ ID {s_id}: {len(juegos)} juegos añadidos.")
        else:
            print(f"❌ ID {s_id}: Error HTTP {response.status_code} (Posible perfil privado)")
            
    except Exception as e:
        print(f"⚠️ ID {s_id}: Error de conexión o formato: {e}")
    
    time.sleep(0.5) # Pausa para evitar el baneo de la API

# Creación del DataFrame y guardado
if registros_juegos:
    df_steam = pd.DataFrame(registros_juegos)
    df_steam.to_csv("./datasets/dataset_steam.csv", index=False)
    print(f"\n¡ÉXITO! Dataset guardado con {len(df_steam)} filas.")
else:
    print("\nNo se pudo extraer información. Revisa tu steam_key.")

--- Iniciando construcción del Dataset (21 usuarios) ---
✅ ID 76561198842603734: 28681 juegos añadidos.
✅ ID 76561198842603734: 28681 juegos añadidos.
✅ ID 76561198023414915: 4646 juegos añadidos.
✅ ID 76561199080934614: 23351 juegos añadidos.
✅ ID 76561197984432884: 3043 juegos añadidos.
✅ ID 76561198254085126: 67 juegos añadidos.
✅ ID 76561198048165534: 15729 juegos añadidos.
✅ ID 76561198023455525: 8895 juegos añadidos.
✅ ID 76561198092430664: 0 juegos añadidos.
✅ ID 76561198294650349: 1344 juegos añadidos.
✅ ID 76561198203118756: 0 juegos añadidos.
✅ ID 76561197986603983: 14692 juegos añadidos.
✅ ID 76561199219841553: 0 juegos añadidos.
✅ ID 76561198212206651: 2818 juegos añadidos.
✅ ID 76561198062673538: 1272 juegos añadidos.
✅ ID 76561198306626714: 0 juegos añadidos.
✅ ID 76561198014898339: 3582 juegos añadidos.
✅ ID 76561198067053149: 3337 juegos añadidos.
✅ ID 76561199499421434: 4257 juegos añadidos.
✅ ID 76561198046160451: 1315 juegos añadidos.
✅ ID 76561198108581917: 0 juegos

In [ ]:
steam_db = pd.read_csv("./datasets/dataset_steam.csv")

In [ ]:
import pandas as pd
import requests
import time

# 2. Función corregida para evitar el KeyError
def obtener_estadisticas_logros(api_key, steam_id, app_id):
    # Endpoint para logros de usuario
    url = "https://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v0001/"
    params = {'key': api_key, 'steamid': steam_id, 'appid': app_id}
    
    try:
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 200:
            data = r.json()
            achievements = data.get('playerstats', {}).get('achievements', [])
            total = len(achievements)
            ganados = sum(1 for a in achievements if a.get('achieved') == 1)
            return ganados, total
    except:
        return 0, 0
    return 0, 0

# 3. Procesar una muestra significativa (ej. los 100 registros más jugados)
# Usamos 'user_id' que es como se guardó en tu CSV anterior
df_sample = steam_db.sort_values(by='playtime_forever', ascending=False).head(100).copy()

logros_data = []
print("Enriqueciendo datos con logros... (esto tardará un poco)")

for idx, row in df_sample.iterrows():
    # CAMBIO CLAVE: Usamos 'user_id' para evitar el KeyError de tu imagen
    ganados, totales = obtener_estadisticas_logros(steam_key, row['user_id'], row['appid'])
    logros_data.append({
        'logros_ganados': ganados,
        'logros_totales': totales,
        'porcentaje_completado': (ganados / totales * 100) if totales > 0 else 0
    })
    time.sleep(0.2) # Pausa de cortesía para la API

# 4. Unir y guardar el dataset final
df_extra = pd.DataFrame(logros_data)
df_final = pd.concat([df_sample.reset_index(drop=True), df_extra], axis=1)

df_final.to_csv("./datasets/dataset_steam_final.csv", index=False)
print("¡Hecho! Dataset final guardado.")

Enriqueciendo datos con logros... (esto tardará un poco)
¡Hecho! Dataset final guardado.


In [26]:
steam_db.head()

,appid,name,playtime_forever,img_icon_url,content_descriptorids,user_id,has_community_visible_stats,has_leaderboards,playtime_2weeks
0,10,Counter-Strike,68,6b0312cda02f5f777efa2f3318c307ff9acafbb5,"[2, 5]",76561198842603734,NaN,NaN,NaN
1,20,Team Fortress Classic,0,38ea7ebe3c1abbbbf4eabdbef174c41a972102b9,"[2, 5]",76561198842603734,NaN,NaN,NaN
2,30,Day of Defeat,0,aadc0ce51ff6ba2042d633f8ec033b0de62091d0,"[2, 5]",76561198842603734,NaN,NaN,NaN
3,40,Deathmatch Classic,0,c525f76c8bc7353db4fd74b128c4ae2028426c2a,NaN,76561198842603734,NaN,NaN,NaN
4,50,Half-Life: Opposing Force,0,04e81206c10e12416908c72c5f22aad411b3aeef,"[2, 5]",76561198842603734,NaN,NaN,NaN


In [24]:
steam_db_final = pd.read_csv("./datasets/dataset_steam_final.csv")

In [25]:
steam_db_final.head(10)

,appid,name,playtime_forever,img_icon_url,content_descriptorids,user_id,has_community_visible_stats,has_leaderboards,playtime_2weeks,logros_ganados,logros_totales,porcentaje_completado
0,366240,GAROU: MARK OF THE WOLVES,1073865,0940266c09476bb48f160be222e370ebed8c5315,NaN,76561198048165534,True,True,6104.0,25,28,89.285714
1,730,Counter-Strike 2,749353,8dbc71957312bbd3baea65848b545be9eae2a355,"[2, 5]",76561199080934614,True,NaN,NaN,1,1,100.000000
2,1267910,Melvor Idle,621639,88eca64c662c128de095b4e452881398ac71969c,NaN,76561198014898339,True,NaN,NaN,86,86,100.000000
3,730,Counter-Strike 2,575615,8dbc71957312bbd3baea65848b545be9eae2a355,"[2, 5]",76561197984432884,True,NaN,1446.0,1,1,100.000000
4,1085660,Destiny 2,533973,fce050d63f0a2f8eb51b603c7f30bfca2a301549,NaN,76561197986603983,True,NaN,NaN,23,23,100.000000
5,730,Counter-Strike 2,487598,8dbc71957312bbd3baea65848b545be9eae2a355,"[2, 5]",76561198023414915,True,NaN,NaN,1,1,100.000000
6,312660,Sniper Elite 4,421042,3a1dbbf59326c0bb78113db7ac0d70b9988b3ce8,"[2, 5]",76561198014898339,True,NaN,NaN,31,85,36.470588
7,1465360,SnowRunner,301058,4ce4d517dc626f3137ffbf5179e797ddb0e7aee0,NaN,76561198014898339,True,NaN,NaN,37,37,100.000000
8,582660,Black Desert,300048,bf5ccace0a692720984827bf042143d0d4b28a42,"[1, 5]",76561198014898339,NaN,NaN,NaN,0,0,0.000000
9,47870,Need for Speed: Hot Pursuit,300032,4066e98d482110d2ee745d3a415d80d36336eb33,NaN,76561198062673538,NaN,NaN,NaN,0,0,0.000000
